# GEMMA-3-1B QLoRA Fine-Tuning for SQL Generation

Complete pipeline: install → load data → 4-bit quantize → LoRA → train → save → test

In [ ]:
# ============================================================================
# INSTALL DEPENDENCIES
# ============================================================================
!pip install -q -U accelerate peft bitsandbytes transformers trl datasets

In [ ]:
# ============================================================================
# IMPORTS & SETUP
# ============================================================================
import os, gc, time, torch, shutil, sys
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainerCallback
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

gc.collect()
torch.cuda.empty_cache()

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
MODEL_NAME        = "google/gemma-3-1b-it"
DATASET_NAME      = "b-mc2/sql-create-context"
OUTPUT_DIR        = "./gemma-sql-finetuned"
NEW_MODEL_NAME    = "gemma-1b-sql-generator"
LOCAL_SAVE_PATH   = f"/content/{NEW_MODEL_NAME}"
NUM_SAMPLES       = 200        # change to 1000 or 2000 for more data
NUM_EPOCHS        = 1          # change to 2 for more epochs
BATCH_SIZE        = 4
GRAD_ACCUM_STEPS  = 4
LEARNING_RATE     = 2e-4
MAX_SEQ_LENGTH    = 256
LORA_R            = 8
LORA_ALPHA        = 16
LORA_DROPOUT      = 0.05
SHUFFLE_SEED      = 42

print("Configuration loaded:")
print(f"  Model:      {MODEL_NAME}")
print(f"  Dataset:    {DATASET_NAME}")
print(f"  Samples:    {NUM_SAMPLES}")
print(f"  Epochs:     {NUM_EPOCHS}")
print(f"  Batch:      {BATCH_SIZE}")
print(f"  LoRA r:     {LORA_R}, alpha: {LORA_ALPHA}")

In [ ]:
# ============================================================================
# LOAD DATASET
# ============================================================================
print(f"Loading dataset: {DATASET_NAME}")
dataset = load_dataset(DATASET_NAME, split="train")
dataset = dataset.shuffle(seed=SHUFFLE_SEED).select(range(NUM_SAMPLES))
print(f"  -> {len(dataset)} samples loaded")

In [ ]:
# ============================================================================
# TOKENIZER
# ============================================================================
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print("  -> pad_token = eos_token")

In [ ]:
# ============================================================================
# FORMAT FUNCTION (Gemma 3 chat template)
# ============================================================================
def format_sql_prompt(example):
    text = f"""<bos><start_of_turn>system
You are an expert SQL assistant. Write only the SQL query, no explanation.<end_of_turn>
<start_of_turn>user
Schema: {example['context']}
Question: {example['question']}<end_of_turn>
<start_of_turn>model
{example['answer']}<end_of_turn><eos>"""
    return {"text": text}

dataset = dataset.map(format_sql_prompt, remove_columns=dataset.column_names)
print(f"Dataset formatted. Sample:{dataset[0]['text'][:100]}...")

In [ ]:
# ============================================================================
# 4-BIT QUANTIZATION CONFIG
# ============================================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
print("BitsAndBytes 4-bit config ready")

In [ ]:
# ============================================================================
# LOAD BASE MODEL
# ============================================================================
print(f"Loading base model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
print("  -> Model loaded in 4-bit")

In [ ]:
# ============================================================================
# LORA CONFIG
# ============================================================================
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)
print(f"LoRA config ready: r={LORA_R}, alpha={LORA_ALPHA}")

In [ ]:
# ============================================================================
# TRAINING ARGUMENTS (SFTConfig)
# ============================================================================
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    logging_steps=1,
    logging_strategy="steps",
    learning_rate=LEARNING_RATE,
    fp16=False,   # must be False with bitsandbytes 4-bit
    bf16=False,   # must be False with bitsandbytes 4-bit
    max_grad_norm=0.3,
    warmup_steps=5,
    lr_scheduler_type="cosine",
    report_to="none",
    save_strategy="epoch",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)
print("SFTConfig ready")

In [ ]:
# ============================================================================
# PROGRESS CALLBACK
# ============================================================================
class ProgressCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.max_steps > 0:
            pct = state.global_step / state.max_steps * 100
            filled = int(20 * state.global_step // state.max_steps)
            bar = "█" * filled + "░" * (20 - filled)
            print(f"\rStep {state.global_step}/{state.max_steps} |{bar}| {pct:.0f}%", end="")
            sys.stdout.flush()
    def on_train_end(self, args, state, control, **kwargs):
        print("\n✅ Training Complete!")

In [ ]:
# ============================================================================
# TRAINER
# ============================================================================
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)
trainer.add_callback(ProgressCallback())

total_steps = trainer.state.max_steps or (
    (len(dataset) // (BATCH_SIZE * GRAD_ACCUM_STEPS)) * NUM_EPOCHS
)

print("=" * 60)
print("TRAINING")
print(f"   Model:    {MODEL_NAME}")
print(f"   Samples:  {len(dataset)}")
print(f"   Epochs:   {NUM_EPOCHS}")
print(f"   Steps:    ~{total_steps}")
print("=" * 60)

In [ ]:
# ============================================================================
# TRAIN
# ============================================================================
start_time = time.time()
trainer.train()
elapsed = time.time() - start_time
print(f"\n⏱️  Training time: {elapsed:.1f}s")

In [ ]:
# ============================================================================
# SAVE MODEL LOCALLY
# ============================================================================
os.makedirs(LOCAL_SAVE_PATH, exist_ok=True)
trainer.model.save_pretrained(LOCAL_SAVE_PATH)
tokenizer.save_pretrained(LOCAL_SAVE_PATH)
print(f"\n💾 Model saved to: {LOCAL_SAVE_PATH}")

for fname in os.listdir(LOCAL_SAVE_PATH):
    fpath = os.path.join(LOCAL_SAVE_PATH, fname)
    if os.path.isfile(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"   📄 {fname} ({size_kb:.1f} KB)")

In [ ]:
# ============================================================================
# SAVE TO GOOGLE DRIVE (Colab only)
# ============================================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BACKUP_PATH = f"/content/drive/MyDrive/{NEW_MODEL_NAME}"
    os.makedirs(DRIVE_BACKUP_PATH, exist_ok=True)
    shutil.copytree(LOCAL_SAVE_PATH, DRIVE_BACKUP_PATH, dirs_exist_ok=True)
    print(f"✅ Saved to Drive: {DRIVE_BACKUP_PATH}")
except Exception as e:
    print(f"⚠️  Drive backup skipped (not in Colab?): {e}")

In [ ]:
# ============================================================================
# SAVE TO HUGGINGFACE HUB (optional)
# ============================================================================
SAVE_TO_HUB = False   # set to True and fill in your HF username
HF_USERNAME = "your-username"

if SAVE_TO_HUB:
    from huggingface_hub import HfApi, login
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    
    api = HfApi()
    repo_id = f"{HF_USERNAME}/{NEW_MODEL_NAME}"
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=LOCAL_SAVE_PATH, repo_id=repo_id, repo_type="model")
    print(f"☁️  Model pushed to Hub: {repo_id}")

In [ ]:
# ============================================================================
# LOAD & TEST THE FINE-TUNED MODEL
# ============================================================================
from peft import PeftModel

# Reload base model (full precision for inference)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Load LoRA adapters
ft_model = PeftModel.from_pretrained(base_model, LOCAL_SAVE_PATH)
ft_tokenizer = AutoTokenizer.from_pretrained(LOCAL_SAVE_PATH)
print("✅ Fine-tuned model loaded for inference")

# Test queries
test_cases = [
    {
        "name": "Simple COUNT",
        "schema": "CREATE TABLE employees (id INT, name TEXT, department TEXT);",
        "question": "How many employees are there?",
    },
    {
        "name": "WHERE clause",
        "schema": "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary REAL);",
        "question": "List all employees in the sales department",
    },
    {
        "name": "JOIN two tables",
        "schema": "CREATE TABLE customers (id INT, name TEXT); CREATE TABLE orders (id INT, customer_id INT, amount REAL);",
        "question": "List customer names with their order amounts",
    },
]

def generate_sql(model, tokenizer, schema, question):
    prompt = f"""<bos><start_of_turn>system
You are an expert SQL assistant. Write only the SQL query, no explanation.<end_of_turn>
<start_of_turn>user
Schema: {schema}
Question: {question}<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.1)
    sql = tokenizer.decode(outputs[0], skip_special_tokens=True)
    result = sql.split("<start_of_turn>model\n")[-1].split("<end_of_turn>")[0].strip()
    return result

print("\n" + "=" * 60)
print("TESTING THE FINE-TUNED MODEL")
print("=" * 60)

for test in test_cases:
    sql = generate_sql(ft_model, ft_tokenizer, test["schema"], test["question"])
    print(f"\n📝 {test['name']}")
    print(f"   SQL: {sql}")

In [ ]:
# ============================================================================
# CLEAR MEMORY
# ============================================================================
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

if torch.cuda.is_available():
    print(f"GPU Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU Cached:    {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print("✅ Memory cleared")